<a href="https://colab.research.google.com/github/DiasFrazerGroup/Intro-to-probabilistic-modelling-2026/blob/main/Session1_Foundations_WithAnswers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 1 — Foundations
*An Introduction to Probabilistic Model Building*

---

This notebook has two parts. Each is self-contained, and you can use the **exercises** at the end of each part if you finish early or want to go deeper.

| Part | Topic | Time |
|---|---|---|
| A | Bayesian basics — the coin flip in code | ~25 min |
| B | Inductive biases — smoothness and sparsity | ~30 min |

---

**The one idea to hold onto throughout:**  
A model is a joint probability distribution. Choosing a prior = encoding what you know before seeing data. Everything downstream is Bayes' theorem:

$$P(\theta \mid \text{data}) \propto P(\text{data} \mid \theta) \; P(\theta)$$

In Part B you'll see how the choice of prior $P(\theta)$ is the central modelling decision — Prior choice will determine if you see a signal or just noise.

## Setup
*Run this first. It takes about 1–2 minutes.*

In [ ]:
!pip install pymc arviz matplotlib numpy scipy --quiet

In [ ]:
import logging

logging.getLogger("pymc").setLevel(logging.ERROR)
logging.getLogger("pytensor").setLevel(logging.ERROR)

import pymc as pm
import arviz as az
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as st

print("PyMC:", pm.__version__)
print("ArviZ:", az.__version__)

In [ ]:
print("Setup complete.")

---
## Part A — Bayesian Basics: the coin flip
*⏱ ~25 min*

We'll build the coin-flip model from the lecture in PyMC.  
Every Bayesian model in PyMC has the same three-step structure:

1. **Prior** — what do we believe about the parameter before seeing data?
2. **Likelihood** — how does the parameter generate data?
3. **Inference** — use MCMC to draw samples from the posterior

The coin-flip model:
- Parameter $\theta$ = bias of the coin (probability of heads)
- Prior: $\theta \sim \text{Beta}(1, 1)$ — uniform, all biases equally plausible
- Likelihood: each flip $y_i \sim \text{Bernoulli}(\theta)$
- Posterior: updated belief about $\theta$ after seeing the flips

In [ ]:
# 10 flips: 7 heads, 3 tails
data = np.array([1]*7 + [0]*3)
print("Observed flips:", data)

In [ ]:
# ⏱ ~30 seconds to run
with pm.Model() as coin_model:
    # 1. Prior
    theta = pm.Beta("theta", alpha=1, beta=1)

    # 2. Likelihood
    y = pm.Bernoulli("y", p=theta, observed=data)

    # 3. Inference — 2 chains is plenty for a 1-parameter model
    idata = pm.sample(2000, tune=1000, chains=2, target_accept=0.9,
                      random_seed=42, return_inferencedata=True)

### What just happened?

`pm.sample` ran the **No-U-Turn Sampler (NUTS)**, a variant of Hamiltonian Monte Carlo.  
It drew 2000 samples from the posterior distribution of θ across 2 independent chains.  

For simple models like this one we *could* compute the posterior analytically  
(it's Beta(1+7, 1+3) = Beta(8, 4) — we did this in the lecture).  
PyMC uses the same machinery for *all* models, including complex ones with no closed-form solution.

**Quick sanity checks:**
- `r_hat` should be very close to 1.00 (chains converged)
- `ess_bulk` should be > 400 (enough effective samples)

In [ ]:
az.summary(idata, var_names=["theta"])

In [ ]:
# Trace plot: each chain should look like a "hairy caterpillar"
az.plot_trace(idata, var_names=["theta"])
plt.tight_layout()
plt.show()

In [ ]:
alpha_prior, beta_prior = 1, 1
alpha_post = alpha_prior + data.sum()
beta_post  = beta_prior  + len(data) - data.sum()

x = np.linspace(0, 1, 200)
prior_pdf     = st.beta.pdf(x, alpha_prior, beta_prior)
posterior_pdf = st.beta.pdf(x, alpha_post,  beta_post)

az.plot_posterior(idata, var_names=["theta"], hdi_prob=0.95)
ax = plt.gca()
ax.plot(x, prior_pdf,     "r--", lw=2, label=f"Prior Beta({alpha_prior},{beta_prior})")
ax.plot(x, posterior_pdf, "g:",  lw=2, label=f"Analytic Beta({alpha_post},{beta_post})")
ax.set_xlabel("Coin bias θ")
ax.legend(fontsize=9)
plt.show()

**What to notice:**
- The MCMC posterior (blue histogram) matches the analytic posterior (green dotted) — good, they should agree.
- The posterior is concentrated around θ ≈ 0.7 (we saw 7 heads out of 10).
- The prior (red dashed) is flat — it contributed almost nothing; 10 flips is already enough to swamp it.

### Posterior predictive check

A model that fits the data should also be able to *simulate* data that looks like what we observed.  
We draw new datasets from the posterior — if the observed outcome (7 heads) falls in a likely region, the model is consistent.

In [ ]:
with coin_model:
    ppc = pm.sample_posterior_predictive(idata, random_seed=42)

sim_heads = ppc.posterior_predictive["y"].sum(axis=-1).values.flatten()

plt.figure(figsize=(7, 4))
plt.hist(sim_heads, bins=np.arange(0, 12) - 0.5, alpha=0.7, label="Simulated data")
plt.axvline(data.sum(), color="red", lw=2, linestyle="--", label=f"Observed: {data.sum()} heads")
plt.xlabel("Heads in 10 flips")
plt.ylabel("Count")
plt.legend()
plt.title("Posterior predictive check")
plt.show()

---
### Exercise A.1 — Prior sensitivity
*⏱ ~5 min*

Copy the coin model cell and change the prior from `Beta(1, 1)` to:
- `Beta(10, 2)` — strongly believes the coin is biased toward heads
- `Beta(2, 10)` — strongly believes the coin is biased toward tails

**Questions:**
1. How does each prior shift the posterior compared to `Beta(1,1)`?
2. How much data would you need to overwhelm the `Beta(10,2)` prior?

In [ ]:
# Your solution here — copy and modify the coin model cell above...or answer using the analytic equations

<details>
<summary>Solution A.1</summary>

- **Beta(10,2):** Prior mean = 10/12 ≈ 0.83 — pulls the posterior upward. With only 10 flips, the posterior is a compromise between prior and data.
- **Beta(2,10):** Prior mean = 2/12 ≈ 0.17 — pulls the posterior downward toward tails.
- A prior of `Beta(10,2)` corresponds to a total of 12 pseudo-observations. So n >> 12.
- **Key insight:** The prior encodes "how many pseudo-observations" you believe in before seeing data. `Beta(α, β)` ≈ α prior heads + β prior tails.
</details>

### Exercise A.2 — Data overwhelms the prior *(optional)*

Keep `Beta(10, 2)` but multiply the data: repeat the 10-flip vector 10× (100 flips), then 100× (1000 flips), keeping the same 70% heads ratio.

**Question:** At what point does the prior become irrelevant?

In [ ]:
# Your solution here


---
## Part B — Inductive Biases in Regression
*⏱ ~30 min total (cells take ~5 min to run — start them and read while they run)*

So far the prior was over a single scalar $\theta$. In regression with many parameters, the prior becomes a rather more interresting design choice.

**Remember:** Inductive bias = prior knowledge baked into the model. We'll look at two biological-relevant types:

| Type | What it encodes | Biological example |
|---|---|---|
| **Smoothness** | Functions shouldn't wiggle more than necessary | Gene expression as a smooth function of dose |
| **Sparsity** | Only a few predictors matter out of many | K active TFs out of hundreds; sparse eQTLs |

In Part B.2, you'll meet the **Horseshoe prior** — the sparsity prior we'll re-encounter in Session 3, where it does something remarkable: compressed sensing.

### B.1 — Smoothness: polynomial regression
*⏱ ~3–4 min to run, then read while it runs*

**The problem with flexible models and small data**

When data is scarce, a flexible model can interpolate perfectly through every
point — fitting noise rather than signal. The standard fix is to pick a simpler
model family (e.g. "use degree 3, not degree 15"), but that requires you to
commit to a specific complexity *before seeing the data*. A prior-based approach
encodes the same belief more honestly: use a maximally flexible model, but tell
it what you expect.

**The model**

We fit a single degree-15 polynomial to all datasets:

$$y_i = \sum_{k=0}^{15} w_k\, x_i^k + \varepsilon_i, \qquad \varepsilon_i \sim \mathcal{N}(0,\,\sigma^2)$$

The coefficients $w_k$ get independent Normal priors, but with
**order-dependent scales**:

$$w_k \sim \mathcal{N}(0,\, s_k^2), \qquad s_k = \begin{cases} 5.0 & k \leq 1 \\ 0.4\, /\, k^{2.5} & k \geq 2 \end{cases}$$

$s_0$ and $s_1$ are large — intercept and slope can be anything. For $k \geq 2$
the allowed amplitude decays rapidly with degree: $s_2 \approx 0.07$,
$s_5 \approx 0.001$. High-degree terms are almost pinned to zero unless the data
demands otherwise.

**Why this encodes smoothness**

A polynomial's wiggles come from its high-degree terms. Shrinking $w_k$ toward
zero for large $k$ means: *the posterior is biased toward smooth functions, but
can deviate from that bias if the evidence is strong enough.* The prior does not
fix the functional form — it shapes a continuous spectrum from "very smooth" to
"wiggly" and lets the likelihood decide where the posterior sits.

**The two scenarios**

- **(a) 5 points from a line** — the data only supports a two-parameter
  description; all higher-order terms are noise. Without the prior, the
  degree-15 polynomial exploits all 15 parameters to interpolate the noise.
  With the prior, the posterior correctly collapses to the linear signal.

- **(b) 18 points from a cubic** — here there *is* genuine nonlinearity in
  the data. The prior for this scenario relaxes the penalty on $k \leq 3$ so
  the cubic structure can be captured, while still suppressing $k \geq 4$
  wiggles. The same prior framework, tuned differently, handles both cases.

**Key takeaway:** you never had to choose a polynomial degree. You picked the
most flexible model and let the prior do the regularisation. It degrades
gracefully to simpler functions when that is all the data supports.

In [ ]:
# ⏱ ~3-4 minutes
import numpy as np
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az
from numpy.polynomial.legendre import legvander  # used for scenario (b)

rng = np.random.default_rng(42)


def standardize(x):
    # Centre and scale x to mean=0, std=1.
    # Essential for polynomial regression: without it, x^15 is astronomically
    # large (or tiny) relative to x^1, making MCMC and numerical arithmetic
    # both unreliable.
    return (x - x.mean()) / x.std()


def poly_design(x, degree):
    # Build the Vandermonde (design) matrix: each row is [1, x_i, x_i^2, ..., x_i^D].
    # Shape: (n_points, degree+1). Column k is the k-th polynomial basis function.
    # The model is then y = X @ w, a linear function of the coefficients w —
    # even though it is nonlinear in x.
    return np.vander(x, N=degree+1, increasing=True)


def posterior_curve_summary(idata, X):
    # Given posterior samples of the coefficient vector w (shape: chains × draws × D+1),
    # compute the implied posterior over the fitted curve at every point in X.
    #
    # Step 1: flatten chains and draws into a single sample dimension → (S, D+1)
    w = idata.posterior["w"].values.reshape(-1, X.shape[1])
    # Step 2: matrix multiply design matrix by all posterior samples at once → (n_grid, S)
    F = X @ w.T
    # Return: posterior mean curve, 5th and 95th percentile (90% credible band)
    return F.mean(1), np.quantile(F, 0.05, 1), np.quantile(F, 0.95, 1)


def fit_poly(X, y, prior_scales, draws=800, tune=800, chains=2, seed=0):
    # Fit a polynomial regression model via MCMC.
    # The key argument is `prior_scales`: a vector of length D+1 giving the
    # prior standard deviation for each coefficient w_k.
    # Everything else is standard PyMC boilerplate.
    with pm.Model():
        # One Normal prior per coefficient. Because sigma is a *vector*,
        # PyMC creates D+1 independent priors — one per polynomial degree.
        # This is where all the inductive bias lives: sigma[k] controls
        # how large the k-th coefficient is allowed to be a priori.
        w = pm.Normal("w", mu=0.0, sigma=prior_scales, shape=X.shape[1])

        # Noise level: HalfNormal(1) is a weakly informative prior that
        # keeps sigma positive and of order ~1 (matching our standardized data).
        sigma = pm.HalfNormal("sigma", sigma=1.0)

        # Likelihood: y_i ~ N(X_i · w, σ²). The dot product X @ w is the
        # polynomial evaluated at each data point.
        pm.Normal("y_obs", mu=pm.math.dot(X, w), sigma=sigma, observed=y)

        return pm.sample(draws, tune=tune, chains=chains, target_accept=0.95,
                         random_seed=seed, return_inferencedata=True)


def orderdep_scales(D, base=5.0, hi=0.4, p=2.5):
    # Prior scales for scenario (a): encode "I expect the function to be smooth".
    #
    # k=0 (intercept) and k=1 (slope): large prior (σ=5), essentially unconstrained.
    # k≥2: scale = hi / k^p, which decays rapidly:
    #   k=2  → 0.4/2^2.5  ≈ 0.07
    #   k=5  → 0.4/5^2.5  ≈ 0.007
    #   k=10 → 0.4/10^2.5 ≈ 0.001
    # Higher-degree terms are almost pinned to zero. The model CAN use them,
    # but only if the data provides very strong evidence that wiggles are real.
    return np.array([base if k <= 1 else hi / k**p for k in range(D + 1)])


def piecewise_scales(D, base=5.0, mid=1.5, hi=0.6, p=2.2, cut=3):
    # Prior scales for scenario (b): encode "I expect up to cubic nonlinearity,
    # but no higher-order wiggles".
    #
    # k=0,1:      σ=5.0  — linear terms unconstrained
    # k=2,3:      σ=1.5  — quadratic/cubic terms allowed to be moderately large
    # k≥4:        σ = hi/k^p — suppressed, same power-law decay as orderdep_scales
    #
    # The only change from orderdep_scales is the relaxed middle band (k=2,3).
    # That is enough for the posterior to capture the cubic signal in the data.
    return np.array([base if k <= 1 else (mid if k <= cut else hi / k**p)
                     for k in range(D + 1)])


# ── (a) 5 points from a line ──────────────────────────────────────────────────
# True signal is linear (y = 0.3 + 1.2x). We deliberately use only 5 points
# so the data is too sparse to constrain 16 free parameters on its own.

Da = 15                                          # polynomial degree
xa = standardize(np.array([0.0, 0.1, 0.55, 0.9, 1.0]))
ya = 0.3 + 1.2*xa + rng.normal(0, 0.2, xa.size) # noisy linear observations
Xa = poly_design(xa, Da)                         # 5×16 design matrix

# Unregularized: every coefficient gets σ=20 — essentially a flat prior.
# With 16 parameters and 5 data points, the system is underdetermined;
# MCMC samples from a very broad posterior, most of which are wildly oscillating.
ida_unreg = fit_poly(Xa, ya, np.full(Da + 1, 20.0), seed=10)

# Order-dependent prior: high-degree terms strongly shrunk.
# The posterior should concentrate near the true linear function.
ida_reg   = fit_poly(Xa, ya, orderdep_scales(Da), seed=11)


# ── (b) 18 points from a cubic ────────────────────────────────────────────────
# True signal is cubic. More data (18 pts) and a relaxed prior on k≤3 terms
# means the posterior should capture the genuine nonlinearity.

Db = 9                                           # degree (lower than (a) — already flexible)
xb = standardize(np.linspace(-1.5, 1.5, 18))
yb_fn = lambda x: 0.5 + 0.8*x - 1.2*x**2 + 0.5*x**3
yb = yb_fn(xb) + rng.normal(0, 0.2, xb.size)

# Legendre basis instead of monomials: Legendre polynomials are orthogonal on
# [-1,1], which makes the design matrix much better conditioned than the raw
# Vandermonde for fitting higher-degree curves.
Xb = legvander(xb, Db)

# Piecewise prior: quadratic and cubic terms allowed; degree≥4 suppressed.
ida_b = fit_poly(Xb, yb, piecewise_scales(Db), seed=12)

print("All models fitted.")

## Sidenote: Why Legendre polynomials??
In the code above we change to the Legendre basis. This is a cool little trick, which we explain here, but feel free to skip as this is not the point of the exercise.

Any degree-9 polynomial can be written in two equivalent ways:

**Monomial basis:**  $y = w_0 + w_1 x + w_2 x^2 + \cdots + w_9 x^9$

**Legendre basis:**  $y = c_0 P_0(x) + c_1 P_1(x) + \cdots + c_9 P_9(x)$

The first few Legendre polynomials are:

$$P_0(x) = 1 \qquad P_1(x) = x \qquad P_2(x) = \frac{3x^2 - 1}{2} \qquad P_3(x) = \frac{5x^3 - 3x}{2}$$

Both bases span exactly the same set of functions. The difference is numerical:
the monomial columns $x^k$ become nearly collinear at higher degrees —
$x^8$ and $x^9$ are almost indistinguishable near $x=1$ — making the design
matrix ill-conditioned and MCMC slow to mix.

Legendre polynomials are orthogonal on $[-1, 1]$:

$$\int_{-1}^{1} P_j(x)\, P_k(x)\, dx = 0 \quad \text{for } j \neq k$$

This means each column of the design matrix is uncorrelated with every other,
giving a well-conditioned matrix and faster sampling. The prior on the
coefficients means the same thing in both bases — higher-index coefficients
control higher-frequency wiggles.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# (a)
ax = axes[0]
ga = np.linspace(xa.min(), xa.max(), 200)
Xga = poly_design(ga, Da)
for idata, color, label in [(ida_unreg, "C3", "Unregularized"), (ida_reg, "C0", "Order-dependent prior")]:
    m, lo, hi = posterior_curve_summary(idata, Xga)
    ax.plot(ga, m, color=color, lw=2, label=label)
    ax.fill_between(ga, lo, hi, color=color, alpha=0.15)
ax.plot(ga, 0.3 + 1.2*ga, "k--", lw=1.5, label="True line")
ax.scatter(xa, ya, s=40, color="k")
ax.set_title("(a) 5 points from a line")
ax.set_ylim(-2, 3); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# (b)
ax = axes[1]
gb = np.linspace(xb.min(), xb.max(), 200)
Xgb = legvander(gb, Db)
m, lo, hi = posterior_curve_summary(ida_b, Xgb)
ax.plot(gb, m, "C0", lw=2, label="Order-dependent prior")
ax.fill_between(gb, lo, hi, color="C0", alpha=0.15)
ax.plot(gb, yb_fn(gb), "k--", lw=1.5, label="True cubic")
ax.scatter(xb, yb, s=40, color="k")
ax.set_title("(b) 18 points from a cubic")
ax.set_ylim(-7, 2); ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle("Smoothness prior: same prior framework, different data", fontsize=12)
plt.tight_layout()
plt.show()

**What to notice:**
- In (a): without the prior, the degree-15 polynomial passes through the 5 points but oscillates wildly between them. With order-dependent shrinkage, it recovers the line.
- In (b): the piecewise prior allows quadratic/cubic terms enough freedom to capture the real curve, while still suppressing degree ≥4 wiggles.
- **The prior didn't pick the model family — it guided how flexible the model is allowed to be.**

---



### B.2 — Sparsity: Normal vs Laplace vs Horseshoe
*⏱ ~4 min to run*

Now a different kind of inductive bias: the belief that **only a few predictors matter**.

Setup: 80 observations, 50 predictors, but only **3 are truly active** (indices 5, 20, 33).  
This is the typical biological setting — thousands of genes, variants, or features, but a handful drive the phenotype.

We'll compare three priors on the regression coefficients:
- **Normal** — no sparsity assumption: all coefficients equally likely to be large or small
- **Laplace** — sharp peak at 0: encourages exact zeros (= frequentist Lasso)
- **Horseshoe** — global–local shrinkage: drives most coefficients to near-zero while letting a few "escape"

> **Looking ahead:** in Session 3 you'll see this Horseshoe prior again — but solving a compressed sensing problem where you have *fewer measurements than unknowns*. The prior is identical; only the design matrix changes.

In [ ]:
# ⏱ ~4 minutes
rng = np.random.default_rng(123)

# ── Synthetic sparse regression problem ───────────────────────────────────────
# 50 observations, 100 candidate predictors — a challenging situation but prior choice can save the day.
n, p = 50, 100                          # underdetermined: more predictors than observations
X_sp = rng.normal(size=(n, p))          # design matrix: each column is one predictor

# Ground truth: only 4 of the 100 predictors actually do anything.
# The remaining coefficients are exactly zero.
# To make things even more challenging, we make one of the signals weak.
beta_true = np.zeros(p)
beta_true[[5, 20, 33, 47]] = [2.5, 0.5, -2.0, 1.8]   # Four active predictorsone with one weak signal at index 20
y_sp = 0.5 + X_sp @ beta_true + rng.normal(0, 1.0, size=n)

y_sp = 0.5 + X_sp @ beta_true + rng.normal(0, 1.0, size=n)


def fit_sparse_model(prior_type, seed=0):
    with pm.Model():
        # Intercept: weakly informative Normal. Not the focus here.
        intercept = pm.Normal("intercept", 0, 2.0)

        if prior_type == "normal":
            # Baseline: independent Normal(0, 2) on every coefficient.
            # This shrinks all coefficients equally toward zero — it has no
            # notion of sparsity. Active and inactive predictors are treated
            # the same, so the posterior spreads probability mass across all 100.
            beta = pm.Normal("beta", 0, 2.0, shape=p)

        elif prior_type == "laplace":
            # Laplace(0, b) = double-exponential prior. Its sharper peak at zero
            # and heavier tails than the Normal mean it shrinks small coefficients
            # more aggressively while allowing a few to be large.
            # This is the Bayesian equivalent of Lasso regression:
            # MAP estimation under a Laplace prior gives exactly the Lasso solution.
            # b=0.5 sets the scale (equivalent to L1 penalty λ = 1/b = 2).
            beta = pm.Laplace("beta", mu=0, b=0.5, shape=p)

        elif prior_type == "horseshoe":
            # The horseshoe prior (Carvalho, Polson & Scott 2010).
            # Each coefficient gets its own local scale λ_j, modulated by a
            # global scale τ shared across all coefficients.
            #
            # τ  ~ HalfCauchy(1): global shrinkage — pulled toward zero when
            #      most predictors are irrelevant, allowing the few active ones
            #      to stand out.
            tau = pm.HalfCauchy("tau", 1.0)
            #
            # λ_j ~ HalfCauchy(1): local scale for predictor j — can escape
            #      the global shrinkage if the data for that predictor is strong.
            #      This is the "horseshoe" shape: a spike at zero (most λ_j small)
            #      with heavy tails (a few λ_j very large).
            lam = pm.HalfCauchy("lam", 1.0, shape=p)
            #
            # β_j ~ Normal(0, τ·λ_j): effective prior scale is τ·λ_j.
            # For inactive predictors: τ small, λ_j small → β_j ≈ 0.
            # For active predictors:  λ_j escapes shrinkage → β_j free.
            # The model learns which is which from the data.
            beta = pm.Normal("beta", 0, tau * lam, shape=p)

        # Noise level: same weakly informative prior across all three models
        # so that any differences in recovery are due to the coefficient prior alone.
        sigma = pm.HalfNormal("sigma", 1.0)

        pm.Normal("y", mu=intercept + pm.math.dot(X_sp, beta),
                  sigma=sigma, observed=y_sp)

        return pm.sample(1000, tune=1000, chains=2, target_accept=0.9,
                         random_seed=seed, return_inferencedata=True)


# Fit all three models to the same data.
# After this cell, compare the posterior distributions of beta across models:
# — Normal:     all 50 posteriors roughly the same width, no clear winners
# — Laplace:    inactive predictors pulled harder toward zero, but still diffuse
# — Horseshoe:  inactive predictors tightly concentrated at zero,
#               active predictors (5, 20, 33) clearly separated from the noise
idata_norm = fit_sparse_model("normal",    seed=1)
idata_lap  = fit_sparse_model("laplace",   seed=2)
idata_hs   = fit_sparse_model("horseshoe", seed=3)

print("All three models fitted.")

In [ ]:
def get_summary(idata):
    mean = idata.posterior["beta"].mean(dim=("chain", "draw")).values
    hdi  = az.hdi(idata, var_names=["beta"], hdi_prob=0.95)["beta"].values
    return mean, hdi

fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
active = np.where(beta_true != 0)[0]

for ax, idata, title, color in [
    (axes[0], idata_norm, "Normal priors",    "C3"),
    (axes[1], idata_lap,  "Laplace priors",   "C1"),
    (axes[2], idata_hs,   "Horseshoe priors", "C0"),
]:
    mean, hdi = get_summary(idata)
    ax.errorbar(np.arange(p), mean,
                yerr=[mean - hdi[:, 0], hdi[:, 1] - mean],
                fmt="o", color=color, ecolor=color, alpha=0.6, capsize=2, ms=3)
    ax.axhline(0, color="k", lw=1)
    ax.scatter(active, beta_true[active], color="black", marker="x", s=100, zorder=5,
               label="True nonzero")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Coefficient index")
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Posterior mean ± 95% HDI")
axes[0].legend(fontsize=9)
plt.suptitle("Sparsity priors: can we recover 4 active coefficients out of 50?", fontsize=12)
plt.tight_layout()
plt.show()

**What to notice:**
- **Normal:** This prior really struggles when the number of observations is less than than the number of features. Wider intervals and signals (×) are hard to distinguish from noise.
- **Laplace:** Inactive coefficients are tightly shrunk toward zero. True signals stand out, but are biased downward and the weak signal at position 20 is lost.
- **Horseshoe:** Inactive coefficients are almost exactly zero. True signals escape shrinkage with less bias — and we get a full posterior (uncertainty) rather than just a point estimate.

The Horseshoe doesn't know in advance which coefficients matter. It learns this from the data — and sets up the kind of analysis you'll do in Session 3.

---
### Exercise B.1 — Laplace shrinkage strength
*⏱ ~5 min*

The current setup is underdetermined: 50 observations, 100 predictors. You have
half as many data points as parameters to estimate, which is why the Normal prior
fails so visibly.

**Part 1 — Shrinkage strength**

In `fit_sparse_model`, change the Laplace scale from `b=0.5` to `b=2.0`
(weaker shrinkage) and refit only the Laplace model.

**Questions:**
1. How does the separation between active and inactive coefficients change?
2. With `b=2.0`, does the Laplace posterior start to resemble the Normal model's?
   Why?
3. What does this tell you about the relationship between the Laplace scale `b`
   and regularisation strength in Lasso regression?

**Part 2 — Adding observations**

Reset `b=0.5`. Now increase `n` from 50 to 200 (still refit only the Laplace
and Normal models to save time) and rerun.

**Questions:**
1. At n=200 the system is overdetermined (2× more observations than predictors).
   How does the Normal model's recovery of the active coefficients change compared
   to n=50?
2. Even at n=200, does the Normal model recover the weak signal (β=0.5 at
   index 20) as cleanly as Laplace? Why or why not?
3. In the n=50 case, the Laplace prior was doing essential work — without it the
   model couldn't identify the active predictors at all. At n=200, the prior is
   less critical. What does this imply about when prior choice matters most
   in practice?

In [ ]:
# Your solution here — copy fit_sparse_model and change b=0.5 to b=2.0


<details>
<summary>Solution B.1</summary>

**Part 1 — Shrinkage strength**

- With `b=2.0`, the Laplace prior is much wider — inactive coefficients no
  longer collapse near zero and the posterior starts to resemble the Normal
  model's: a broad spread of small but nonzero estimates across all 100
  predictors.
- This is because the Laplace scale `b` = 1/λ in Lasso notation. Large `b`
  → small λ → weak regularisation. With an underdetermined system (n=50,
  p=100), weak regularisation is catastrophic: without a strong push toward
  zero, the model has no way to distinguish signal from noise.
- **Key insight:** the *type* of prior (Laplace vs Normal) determines the
  *shape* of shrinkage (sparse vs uniform). The *scale* (`b`, `sigma`)
  determines its *strength*. In an underdetermined regime, both matter — a
  sparse prior with too large a scale is no better than a Normal prior.

**Part 2 — Adding observations**

- At n=200 the Normal model improves substantially: the active coefficients
  (2.5, −2.0, 1.8) are now visible above the noise floor. With more data
  than parameters, the likelihood itself resolves most of the ambiguity, and
  the prior is less load-bearing.
- The weak signal (β=0.5) is still harder for the Normal model to recover
  cleanly. Equal shrinkage pulls it toward zero by the same amount as the
  inactive predictors, so it blurs into the background. Laplace's sharper
  peak at zero shrinks the inactive predictors more aggressively, giving the
  0.5 signal more room to stand out.
- **Key insight:** prior choice matters most when data is scarce relative to
  model complexity (small n, large p). As n grows the likelihood dominates
  and even a poor prior gets washed out.
</details>

### Exercise B.2 — More active predictors *(optional)*

Change `beta_true[[5, 20, 33]] = [2.0, -1.5, 1.2]` to activate 15 coefficients instead of 3.  
Keep N=80, P=50.

**Questions:**
1. Does the Horseshoe still separate signal from noise?
2. At what point does the sparsity assumption break down?
3. What would be a better prior when the signal is dense (many active predictors)?

In [ ]:
# Your solution here


---

## Where we're going next

| Session | Notebook | What today's ideas become |
|---------|----------|--------------------------|
| **2a** | Hierarchical Models | The prior/posterior update you just did for θ now happens for K=12 clinical trials simultaneously; each trial's effect is a latent variable with a *shared* population prior |
| **2b** | Protein VAE | The ELBO — reconstruction minus KL — is the same objective as the coin-flip model, but p(x|z) is a neural network and z is a continuous latent code per sequence |
| **3a** | BOED | The posterior you built in Part A is exactly what the adaptive loop queries each round: "where is p(θ|data) still wide?" is the same as "where should I measure next?" |
| **3b** | Compressed Sensing | The Horseshoe prior from Part B is the *enabling ingredient* for pooled ELISPOT design — sparsity is what lets you recover K=5 epitopes from M=30 mixed pools instead of 96 individual wells |

The prior you choose = the inductive bias you impose.  
Today you saw this for smoothness (polynomial regression) and sparsity (Horseshoe).  
In Sessions 2 and 3 the same principle governs neural-network likelihoods, variant scoring, experiment choice, and epitope discovery.